# Peut-on prédire le succès d'une startup dès sa première phrase de pitch ?

*Capucine, Gaspard, Romain — Python pour la data science*

Ce notebook s'exécute de la première à la dernière cellule sans erreur, en repartant des copies
datées des données brutes stockées sur S3. Les fonctions sont dans `src/`.

GitHub n'affiche pas les graphiques interactifs (Plotly) : pour les voir, lire le rapport sur
[nbviewer](https://nbviewer.org/github/romainmugniot/projet-pitch-startups/blob/main/rapport.ipynb).

## Introduction

Y Combinator (YC), le plus connu des accélérateurs de startups, demande à chaque équipe candidate de
décrire son entreprise en quelques mots. Cette phrase, le pitch, est aussi la première chose que lit
un investisseur qui trie des centaines de dossiers. Elle obéit à des conseils bien établis (nommer
son client, donner un chiffre, éviter le jargon) et suit des modes : « the Uber for X » au milieu
des années 2010, « AI-powered » aujourd'hui. Ce rapport se demande si cette première phrase dit
quelque chose du destin de la startup : celles qui échouent se présentaient-elles autrement que
celles qui durent ?

Deux hypothèses s'opposent. Selon la première, le pitch révèle la clarté du projet : une équipe qui
sait dire précisément ce qu'elle fait et pour qui aurait plus de chances de réussir, et le jargon
trahirait un projet flou. Selon la seconde, le pitch ne fait que refléter le secteur et l'époque :
un pitch grand public ou chargé de mots à la mode signale un secteur plus risqué ou une année
donnée, et, une fois ces deux facteurs pris en compte, il n'apporte plus rien.

Pour trancher, on relie le pitch d'origine de 392 startups YC des promotions 2017 à 2021, publié
sur Hacker News à leur lancement, à leur statut actuel dans la base YC. La partie 1 construit ces
données et justifie le choix du pitch d'origine plutôt que du pitch actuel. La partie 2 définit
l'échec et mesure ce que le secteur et la promotion expliquent déjà. La partie 3 décrit les pitchs
et les compare à l'échec à promotion égale. La partie 4 refait la comparaison à promotion et
secteur égaux, puis vérifie qu'elle ne dépend pas des choix de la partie 1.

## Paramètres

`COLLECTER = False` : le notebook lit la copie des données du 30/10/2026 sur le stockage S3 du SSP Cloud, celle que commente le rapport. Avec `COLLECTER = True`, il retélécharge les données depuis les API : les chiffres peuvent alors différer un peu du texte.

In [1]:
import pandas as pd
import plotly.io as pio

from src import analyse, collecte, graphiques, nettoyage, succes, texte
from src.config import DATE_COPIE, DICTIONNAIRE_VARIABLES

COLLECTER = False  # True : données à jour depuis les API au lieu de la copie du DATE_COPIE
GRAINE = 2026  # rend reproductible le tirage des 50 appariements vérifiés à la main (partie 1.3)

# Graphiques Plotly visibles dans VSCode ou Jupyter comme dans la version en ligne (nbviewer).
pio.renderers.default = "plotly_mimetype+notebook_connected"

texte.preparer_nltk()
print(f"Copie des données : {DATE_COPIE} ; collecte depuis les API : {COLLECTER}")

Copie des données : 2026-10-30 ; collecte depuis les API : False


## Données

La base Y Combinator et les posts Launch HN sont appariés pour former la table commune : une ligne par startup qui a publié un Launch HN, avec son pitch d'origine et son statut actuel.

In [2]:
yc = collecte.charger_brut("yc", collecter=COLLECTER)
hn = collecte.charger_brut("hn", collecter=COLLECTER)
table = nettoyage.joindre(yc, hn)

print(f"Base YC : {len(yc)} startups ; posts Launch HN : {len(hn)} ; "
      f"table commune : {len(table)} startups appariées")
table.head()

Base YC : 6354 startups ; posts Launch HN : 1159 ; table commune : 853 startups appariées


,slug,nom,promotion,pitch_origine,pitch_actuel,statut,niveau_succes,secteur,region,taille_equipe,stage,top_company,anciens_noms,hn_date,hn_points,hn_commentaires,hn_delai_ans
0,sleep-reset,Sleep Reset,Winter 2017,Spotify for Meditation,One clinic for every sleep issue,Active,<NA>,Healthcare,America / Canada,8.0,Growth,False,[Simple Habit],2017-01-10 17:48:24+00:00,118,86,0.024641
1,scaphold-io,Scaphold.io,Winter 2017,GraphQL Backend as a Service,The Universal API,Acquired,<NA>,B2B,America / Canada,2.0,Early,False,[],2017-01-24 18:27:08+00:00,143,93,0.062971
2,upkeep,UpKeep,Winter 2017,Basecamp for Facility Maintenance Teams,Mobile-first software for maintenance teams.,Acquired,<NA>,B2B,America / Canada,150.0,Growth,False,[UpKeep Maintenance Management],2017-01-25 18:02:00+00:00,80,60,0.065708
3,voodoo-manufacturing,Voodoo Manufacturing,Winter 2017,AWS for Manufacturing,AWS for manufacturing.,Inactive,<NA>,Industrials,America / Canada,19.0,Early,False,[],2017-01-26 18:00:33+00:00,166,59,0.068446
4,peer5,Peer5,Winter 2017,Serverless CDN,P2P delivery network that enables high quality...,Acquired,<NA>,B2B,America / Canada,11.0,Early,False,[],2017-01-27 18:37:58+00:00,108,117,0.071184


## 1. Construire les données

Aucune base ne relie directement le pitch d'une startup à son destin : on la construit en appariant
deux sources publiques, puis on restreint l'étude aux startups dont on connaît vraiment l'issue.

### 1.1 Deux sources complémentaires

| Source | Ce qu'on en tire | Accès |
|---|---|---|
| Base Y Combinator (API yc-oss) | Une ligne par startup : nom, anciens noms, promotion, statut actuel (*Active*, *Inactive*, *Acquired*, *Public*), secteur, région, taille de l'équipe, stade, pitch actuel (`one_liner`) | Fichier JSON public, sans clé |
| Posts « Launch HN » (API Algolia de Hacker News) | Une ligne par post : titre « Launch HN: Nom (YC W20) – pitch », date, points, commentaires | API de recherche publique, 1 000 résultats par requête, pagination par date |

Y Combinator donne le destin de chaque startup, Hacker News donne la phrase avec laquelle elle s'est
présentée au public. Les deux copies analysées datent du 30/10/2026 (`DATE_COPIE`).

In [3]:
promotions_yc = yc["batch"].map(nettoyage.debut_promotion)
dates_hn = pd.to_datetime(hn["created_at"])
pd.DataFrame(
    {
        "lignes": [len(yc), len(hn)],
        "unité": ["startup", "post"],
        "de": [promotions_yc.min().year, dates_hn.min().year],
        "à": [promotions_yc.max().year, dates_hn.max().year],
    },
    index=["Base Y Combinator (promotions)", "Posts Launch HN (dates)"],
)

,lignes,unité,de,à
Base Y Combinator (promotions),6354,startup,2005,2027
Posts Launch HN (dates),1159,post,2009,2026


La base YC couvre toutes les promotions depuis 2005, alors que les Launch HN n'existent en nombre
qu'à partir de la fin des années 2010 : c'est Hacker News qui limite la taille de l'échantillon.
Seule une partie des startups YC publie un Launch HN, ce qui pose la question de la
représentativité (section 1.5).

### 1.2 Pourquoi le pitch d'origine

La base YC contient bien une phrase de présentation, le `one_liner`, mais c'est celle d'aujourd'hui :
une startup la réécrit à mesure qu'elle pivote ou grandit. Mettre en regard ce pitch actuel et le
succès serait circulaire, puisqu'une startup qui a réussi décrit ce qu'elle est devenue. Le titre du
post Launch HN, publié en général quelques mois après le début de la promotion, fige au contraire la
phrase des débuts.

In [4]:
exemples = ["Substack", "PostHog", "Hightouch", "Vapi"]
display(
    table.loc[
        table["nom"].isin(exemples), ["nom", "promotion", "statut", "pitch_origine", "pitch_actuel"]
    ]
)

def _simplifier(pitchs: pd.Series) -> pd.Series:
    return pitchs.str.lower().str.strip(" .")

identiques = _simplifier(table["pitch_origine"]) == _simplifier(table["pitch_actuel"])
print(f"Pitch actuel identique au pitch d'origine : {identiques.sum()} startups sur {len(table)} "
      f"({identiques.mean():.0%})")

,nom,promotion,statut,pitch_origine,pitch_actuel
57,Substack,Winter 2018,Active,Paid email newsletters made simple,A place for independent writing.
137,Hightouch,Summer 2019,Active,We Book Travel for You on Slack,The leading Composable CDP and AI Decisioning ...
177,PostHog,Winter 2020,Active,open-source product analytics,"The single platform to analyze, test, observe,..."
316,Vapi,Winter 2021,Active,Calendar in your menu bar,Voice AI for developers.


Pitch actuel identique au pitch d'origine : 28 startups sur 853 (3%)


Seuls 3 % des pitchs n'ont pas changé, à la casse et à la ponctuation près. Les réécritures vont de la
reformulation (Substack : « Paid email newsletters made simple » devient « A place for independent
writing ») au pivot complet (Carry, qui réservait des voyages sur Slack, est devenue Hightouch, une
plateforme de données clients). C'est donc bien le pitch d'origine qu'il faut étudier.

### 1.3 Collecte et jointure

`collecte.telecharger_launch_hn` récupère tous les posts du tag `launch_hn`, par paquets de 1 000 en
remontant le temps. `nettoyage.extraire_titre` découpe chaque titre en nom, promotion (« W20 » devient
« Winter 2020 ») et pitch. `nettoyage.joindre` apparie ensuite chaque post à la base YC sur le nom
normalisé (minuscules, sans accents ni suffixes comme « Inc » ou « AI ») et la promotion, puis, pour
les startups renommées, sur leurs anciens noms. Une startup lancée deux fois garde son post le plus
ancien.

Les posts dont le titre n'indique pas de promotion sont écartés : sans elle, deux startups homonymes
de promotions différentes seraient confondues, et le risque d'apparier un post à la mauvaise
entreprise est plus grave que la perte de ces posts.

In [5]:
bilan = nettoyage.bilan_jointure(table, hn)
sans_promotion = int(bilan["posts"].get("inconnue", 0))
non_reconnus = int(hn["title"].map(lambda titre: nettoyage.extraire_titre(titre)["nom"] is None).sum())
avec_promotion = len(hn) - sans_promotion
print(f"{len(hn)} posts Launch HN, dont {sans_promotion} sans promotion dans le titre "
      f"({non_reconnus} au format non reconnu) : écartés")
print(f"{len(table)} startups appariées : {len(table) / len(hn):.0%} des posts, "
      f"{len(table) / avec_promotion:.0%} des posts avec promotion")

par_annee = (
    bilan.drop(index="inconnue", errors="ignore")
    .groupby(lambda promotion: int(promotion.split()[1]))[["posts", "appariements"]]
    .sum()
    .rename_axis("année de promotion")
)
par_annee["taux"] = (par_annee["appariements"] / par_annee["posts"]).round(2)
par_annee.T

1159 posts Launch HN, dont 222 sans promotion dans le titre (94 au format non reconnu) : écartés
853 startups appariées : 74% des posts, 91% des posts avec promotion


année de promotion,2008,2012,2014,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
posts,1.0,1.0,2.0,4.0,64.00,56.00,101.00,111.0,184.00,104.00,85.00,86.0,77.0,61.0
appariements,1.0,0.0,1.0,2.0,61.00,52.00,90.00,100.0,168.00,97.00,80.00,77.0,69.0,55.0
taux,1.0,0.0,0.5,0.5,0.95,0.93,0.89,0.9,0.91,0.93,0.94,0.9,0.9,0.9


Trois posts sur quatre aboutissent à une startup (74 %). L'essentiel des pertes vient des 222 posts
sans promotion lisible ; parmi les posts qui en ont une, 91 % sont appariés, et une partie du reste
correspond à des startups lancées deux fois, qui ne comptent qu'une fois.

#### Vérification manuelle de 50 appariements

Un appariement automatique peut se tromper sans bruit : un homonyme dans la même promotion, ou un
ancien nom partagé. On tire donc 50 startups de la table commune (graine fixe) et on compare, à la
main, le nom du post Launch HN, le nom dans la base YC, le pitch d'origine et le pitch actuel.

In [6]:
titres = (
    pd.DataFrame(hn["title"].map(nettoyage.extraire_titre).tolist())
    .dropna()
    .drop_duplicates(subset=["promotion", "pitch"])
    .rename(columns={"nom": "nom_launch_hn", "pitch": "pitch_origine"})
)
verification = table.sample(50, random_state=GRAINE).merge(
    titres, on=["promotion", "pitch_origine"], how="left"
)
with pd.option_context("display.max_rows", 50, "display.max_colwidth", 70):
    display(verification[["nom_launch_hn", "nom", "promotion", "pitch_origine", "pitch_actuel"]])

,nom_launch_hn,nom,promotion,pitch_origine,pitch_actuel
0,Fastgen,Fastgen,Winter 2023,Visual Low-Code Backend Builder,Webflow for the Backend
1,Type,Type,Winter 2023,AI-powered document editor,AI-native storytelling studio.
2,Riot,Riot,Winter 2020,Phishing training for your team,Cybersecurity awareness for fast growing companies
3,Recall.ai,Recall.ai,Winter 2020,API for meeting recordings and transcripts,"The API to get recordings, transcripts, and metadata from meetings"
4,Matrubials,Matrubials,Summer 2021,Milk-derived therapeutics for infectious diseases,Developing milk-inspired therapeutics for infectious diseases
5,Magic Patterns,Magic Patterns,Winter 2023,AI Design and Prototyping for Product Teams,The AI-native workspace to design your product.
6,Narrator,Narrator,Summer 2019,a data modeling platform built on a single table,The only platform that can prep data to answer EVERY question you…
7,Firezone,Firezone,Winter 2022,Zero-trust access platform built on WireGuard,Easiest way to set up Zero Trust Access for your team
8,AirMyne,AirMyne,Winter 2022,Capturing CO2 from air at industrial scale,Building systems to remove carbon dioxide from air at industrial s...
9,Baselit,Retape,Winter 2023,Automatically Reduce Snowflake Costs,AI Operating System for Debt Settlement


Sur la copie du 30/10, les 50 appariements sont corrects : 44 portent le même nom des deux côtés, à
l'écriture près (« Axdraft » et « AXDRAFT », « Nao Labs » et « nao Labs »), et les 6 autres sont des
startups renommées que la seconde passe retrouve par leur ancien nom, confirmé dans la base YC
(Baselit devenue Retape, Bend devenue Versa, Realize devenue GovDash, Paigo devenue Champ, Financial
Choice devenue Nexus, Tersho devenue Slidely AI). Zéro erreur sur 50 ne prouve pas qu'il n'y en a
aucune, mais situe le taux d'erreur sous 7 % avec 95 % de confiance (borne haute de l'intervalle de
Wilson). Une première vérification sur l'échantillon d'étude a relevé un autre défaut, qui touche le
pitch et non l'appariement : le titre « Algofi (YC S21) Lofty (YC S19) Crypto as Collateral to Buy
Property », sans séparateur, donne un pitch qui commence par le nom d'une autre startup. Ce cas reste
isolé, mais il est dans l'échantillon d'étude et c'est un échec : son pitch compte à tort comme
contenant un chiffre (« S19 ») et un mot à la mode (« crypto »), ce qui pèse sur ces deux traits rares
(partie 3.3).

### 1.4 Période d'étude et échantillon final

Le statut *Inactive* n'a de sens qu'avec du recul : une startup de la promotion 2025 n'a pas encore
eu le temps d'échouer. Le graphique compare la part d'inactives par année de promotion dans toute la
base YC et dans la table commune.

In [7]:
graphiques.courbe_recul(yc, table)

Dans la base YC, la part d'inactives tourne autour de 30 % jusqu'en 2016, puis baisse presque chaque
année jusqu'à 1 ou 2 % pour les promotions 2025 et 2026, et la table commune suit la même pente : sur
les promotions récentes, comparer des taux d'échec reviendrait à comparer des âges. On garde les
promotions 2017 à 2021, qui ont au moins cinq ans de recul ; avant 2017, les Launch HN sont trop rares
(8 posts). Même sur cette fenêtre, la part d'inactives baisse encore (28 % pour 2017 contre 18 % pour
2021 dans la base YC) : c'est pourquoi les comparaisons suivantes se font aussi à promotion égale.

In [8]:
etude, etapes = nettoyage.echantillon_etude(table, debut=2017, fin=2021, delai_max=1.0)
etapes.round(3)

,startups,echecs,part_echecs
etape,,,
Table commune,853,136,0.159
Promotions 2017 à 2021,471,104,0.221
Post Launch HN au plus 1 an après la promotion,392,90,0.230


In [9]:
annees = table["promotion"].map(nettoyage.debut_promotion).dt.year
periode = table[annees.between(2017, 2021)]
moment = (
    (periode["hn_delai_ans"] > 1)
    .map({False: "Post dans l'année", True: "Post plus d'un an après"})
    .rename("moment du post")
)
statuts = pd.crosstab(moment, periode["statut"], normalize="index").mul(100).round(0)
statuts.insert(0, "startups", moment.value_counts())
print(f"Délai médian entre le début de la promotion et le post : "
      f"{etude['hn_delai_ans'].median() * 12:.1f} mois")
statuts

Délai médian entre le début de la promotion et le post : 2.3 mois


statut,startups,Acquired,Active,Inactive
moment du post,,,,
Post dans l'année,392,22.0,55.0,23.0
Post plus d'un an après,79,24.0,58.0,18.0


La plupart des startups publient leur Launch HN pendant ou juste après la promotion (délai médian
d'un peu plus de deux mois). On écarte les 79 posts publiés plus d'un an après, pour deux raisons :
leur pitch n'est plus celui des débuts, et seules les startups encore en vie peuvent se lancer
tardivement. Ces posts tardifs comptent d'ailleurs moins d'inactives (18 % contre 23 %) : les garder
gonflerait artificiellement la survie. L'échantillon d'étude compte 392 startups, dont 90 échecs
(23 %).

### 1.5 Représentativité

Les startups qui font un Launch HN ne sont pas tirées au hasard parmi les startups YC. On compare
l'échantillon à toutes les startups YC des promotions 2017 à 2021 (pourcentages).

In [10]:
yc_periode = yc[promotions_yc.dt.year.between(2017, 2021)]
print(f"Startups YC 2017-2021 : {len(yc_periode)} ; échantillon : {len(etude)} "
      f"({len(etude) / len(yc_periode):.0%})")


def comparer(colonne_yc: str, colonne_etude: str) -> pd.DataFrame:
    """Répartition (en %) d'une variable dans la base YC 2017-2021 et dans l'échantillon."""
    return (
        pd.concat(
            {
                "YC 2017-2021": yc_periode[colonne_yc].value_counts(normalize=True),
                "Échantillon": etude[colonne_etude].value_counts(normalize=True),
            },
            axis=1,
        )
        .fillna(0)
        .mul(100)
        .round(1)
    )


display(comparer("status", "statut"))
display(comparer("industry", "secteur"))

Startups YC 2017-2021 : 2049 ; échantillon : 392 (19%)


,YC 2017-2021,Échantillon
Active,60.8,54.8
Inactive,22.8,23.0
Acquired,16.3,22.2
Public,0.2,0.0


,YC 2017-2021,Échantillon
B2B,42.3,54.6
Healthcare,15.8,8.9
Consumer,14.9,15.8
Fintech,13.1,9.9
Industrials,6.7,5.9
Education,3.2,3.1
Real Estate and Construction,3.0,1.3
Government,0.9,0.5
Unspecified,0.1,0.0


L'échantillon représente environ une startup YC sur cinq de la période. Son taux d'échec est le même
que celui de la base (23 %), ce qui écarte un biais grossier sur la variable étudiée ; les rachats y
sont un peu plus fréquents (22 % contre 16 %). Le secteur est plus déformé : le B2B, dont les
fondateurs visent un public de développeurs très présent sur Hacker News, pèse 55 % de l'échantillon
contre 42 % dans YC, la santé deux fois moins (9 % contre 16 %). Les résultats valent donc pour
« les startups YC qui ont fait un Launch HN », surtout B2B, et pas pour toutes les startups.

## 2. Qui réussit ? Définir et décrire le succès

### 2.1 Définir la réussite

La variable étudiée dans tout le rapport est l'échec : une startup a échoué si la base YC la dit
*Inactive* (`succes.echec`). C'est la seule information de destin disponible pour toutes les
startups et sans ambiguïté. Pour nuancer, `succes.niveau_succes` classe aussi les startups en quatre
niveaux : échec, survie (active), croissance (active, au stade *Growth* ou avec une équipe plus
grande que la médiane des actives de sa promotion) et sortie (rachetée ou en bourse). Le seuil
d'équipe dépend de la promotion parce qu'une startup plus ancienne a eu plus de temps pour embaucher.

In [11]:
etude["echec"] = succes.echec(etude)
etude["niveau_succes"] = succes.niveau_succes(etude)

niveaux = etude["niveau_succes"].value_counts(sort=False).to_frame("startups")
niveaux["part (%)"] = (niveaux["startups"] / len(etude) * 100).round(0)
display(niveaux)

seuils = etude["taille_equipe"].where(etude["statut"] == "Active").groupby(etude["promotion"]).median()
sauvetage = succes.rachats_de_sauvetage(etude)
print(f"Seuil d'équipe de la croissance : de {seuils.min():.0f} à {seuils.max():.0f} personnes "
      "selon la promotion")
print(f"Rachats de sauvetage (5 personnes ou moins) : {sauvetage.sum()} "
      f"sur {(etude['statut'] == 'Acquired').sum()} rachats")

,startups,part (%)
niveau_succes,,
échec,90,23.0
survie,93,24.0
croissance,122,31.0
sortie,87,22.0


Seuil d'équipe de la croissance : de 10 à 80 personnes selon la promotion
Rachats de sauvetage (5 personnes ou moins) : 32 sur 87 rachats


Les quatre niveaux ont des tailles comparables (de 87 à 122 startups). Le seuil d'équipe va de
10 personnes (Summer 2021, la promotion la plus récente) à 80 (Winter 2018) et il est en général plus
haut pour les promotions anciennes, ce qui justifie de le calculer par promotion. Plus d'un rachat sur trois (32 sur 87) concerne une équipe de cinq personnes
ou moins : ces rachats ressemblent plus à un recrutement ou à une sortie de secours qu'à une réussite.
C'est une raison de plus pour s'en tenir à l'échec, qui ne dépend pas de cette frontière, comme
variable principale.

### 2.2 Contrôles qualité

In [12]:
succes.controles_qualite(etude).round(3)

,nombre,part
controle,,
Statut manquant,0,0.000
Équipe manquante ou à 0,10,0.026
Slug en double,0,0.000
Région manquante,5,0.013
Promotion non reconnue,0,0.000


Le statut, dont dépend l'échec, ne manque jamais, aucune startup n'apparaît deux fois et toutes les
promotions sont reconnues. Les seuls trous concernent la taille d'équipe (3 % des startups, qui ne
peuvent alors être classées en croissance que par leur stade) et la région (1 %, regroupées en
« Non renseigné »).

### 2.3 Taux d'échec par secteur et par région

Chaque barre donne la part de startups inactives et son intervalle de confiance à 95 % (Wilson) ;
l'effectif est entre parenthèses, et les secteurs de moins de 10 startups sont regroupés en
« Autres ».

In [13]:
taux_secteur = analyse.taux_echec(etude, "secteur")
graphiques.barres_taux(
    taux_secteur, "Les startups grand public échouent deux fois plus souvent que la moyenne"
)

In [14]:
analyse.taux_echec_standardise(etude, "secteur").round(2)

,effectif,observes,attendus,ratio,ic_bas,ic_haut
secteur,,,,,,
Consumer,62,32,14.62,2.19,1.50,3.09
Industrials,23,6,5.08,1.18,0.43,2.57
B2B,214,40,48.83,0.82,0.59,1.12
Fintech,39,6,8.65,0.69,0.25,1.51
Healthcare,35,3,8.34,0.36,0.07,1.05
Education,12,0,2.50,0.00,0.00,1.46
Autres,7,3,1.98,1.51,0.30,4.43


Le secteur grand public (*Consumer*) se détache : 52 % d'échecs, contre 23 % pour l'ensemble, et son
intervalle ne recoupe pas celui du B2B (19 %). À promotion égale (tableau, standardisation
indirecte), il compte 2,2 fois plus d'échecs qu'attendu, avec un intervalle de 1,5 à 3,1 : l'écart
ne vient ni du hasard ni de l'âge des startups. À l'autre bout, la santé échoue peu (9 %, ratio de
0,4), et l'éducation ne compte aucun échec, mais sur 12 startups seulement.

In [15]:
taux_region = analyse.taux_echec(etude, "region")
graphiques.barres_taux(taux_region, "Hors Amérique du Nord, les effectifs sont trop faibles pour conclure")

Trois startups sur quatre de l'échantillon sont nord-américaines (293 sur 392). Leur taux d'échec
(25 %) est un peu plus haut que celui des startups européennes (14 %), mais les intervalles se
recouvrent largement, et les autres régions comptent moins de 15 startups chacune : la région ne
distingue pas les startups de façon fiable. On ne la retient pas dans la suite : avec trois startups
sur quatre en Amérique du Nord, elle découperait les cases promotion × secteur sans rien apporter.

### 2.4 La barre à dépasser

In [16]:
taux_promotion = analyse.taux_echec(etude, "promotion")
secteurs = taux_secteur.drop(index=[analyse.AUTRES, analyse.NON_RENSEIGNE], errors="ignore")
for nom, taux in [("Secteur", secteurs), ("Promotion", taux_promotion)]:
    haut, bas = taux["taux"].idxmax(), taux["taux"].idxmin()
    print(f"{nom} : de {taux.loc[bas, 'taux']:.0%} ({bas}, {taux.loc[bas, 'effectif']} startups) "
          f"à {taux.loc[haut, 'taux']:.0%} ({haut}, {taux.loc[haut, 'effectif']} startups)")

Secteur : de 0% (Education, 12 startups) à 52% (Consumer, 62 startups)
Promotion : de 5% (Winter 2019, 40 startups) à 42% (Winter 2018, 26 startups)


Sans lire un seul mot du pitch, le secteur fait déjà passer le taux d'échec de 0 à 52 %. L'écart entre
promotions, de 5 % (Winter 2019) à 42 % (Winter 2018), tient en grande partie au hasard, sur 40 et
26 startups : dans toute la base YC, la part d'inactives baisse plus régulièrement, de 28 % pour 2017
à 18 % pour 2021 (partie 1.4). Il reste que l'âge de la promotion compte. Un mot qui paraît associé à l'échec peut donc
simplement signaler un secteur risqué (« app », « consumers ») ou une promotion plus ancienne. C'est la
barre à dépasser : dans les parties 3 et 4, un trait du pitch ne compte que s'il reste associé à
l'échec à promotion et secteur égaux.

## 3. Que disent les pitchs ?

On étudie le pitch d'origine, celui du post Launch HN. Comme le taux d'échec dépend de la promotion
(partie 2), les traits du pitch, sa réécriture et la réaction de Hacker News sont comparés à l'échec
à promotion égale. La recherche mot par mot (3.4) compare des effectifs bruts : chaque mot apparaît
dans trop peu de pitchs pour être découpé par promotion.

### 3.1 Anatomie d'un pitch

`texte.indicateurs` mesure, pour chaque pitch, cinq traits simples : sa longueur, la présence d'un
chiffre (« in 24 hours »), d'un client nommé après « for » (« for restaurants »), d'une tournure
« X for Y » (« Uber for… », « the Stripe for… », repérée à partir d'une liste de marques connues)
et d'un mot à la mode tiré de la liste `texte.BUZZWORDS` (« AI-powered », « seamless »…).

In [17]:
indic = texte.indicateurs(etude["pitch_origine"])
longueur_mediane = indic["nb_mots"].median()
etude = etude.assign(**texte.traits_pitch(etude["pitch_origine"], longueur_mediane))

INDICATEURS = {
    "chiffre": "Contient un chiffre",
    "client_nomme": "Nomme un client (« for … »)",
    "x_for_y": "Tournure « X for Y »",
    "jargon": "Contient un mot à la mode",
    "long": f"Plus de {longueur_mediane:.0f} mots",
}
anatomie = pd.DataFrame(
    {
        "startups": [int(etude[colonne].sum()) for colonne in INDICATEURS],
        "part (%)": [round(etude[colonne].mean() * 100, 1) for colonne in INDICATEURS],
        "exemple": [etude.loc[etude[colonne], "pitch_origine"].iloc[0] for colonne in INDICATEURS],
    },
    index=pd.Index(list(INDICATEURS.values()), name="indicateur"),
)
print(f"Longueur médiane : {longueur_mediane:.0f} mots ({indic['nb_caracteres'].median():.0f} "
      f"caractères) ; la moitié des pitchs fait entre {indic['nb_mots'].quantile(0.25):.0f} et "
      f"{indic['nb_mots'].quantile(0.75):.0f} mots")
anatomie

Longueur médiane : 6 mots (39 caractères) ; la moitié des pitchs fait entre 5 et 7 mots


,startups,part (%),exemple
indicateur,,,
Contient un chiffre,14,3.6,401(k) for Healthcare
Nomme un client (« for … »),130,33.2,Spotify for Meditation
Tournure « X for Y »,8,2.0,Spotify for Meditation
Contient un mot à la mode,8,2.0,AI-Powered Sensors for Building Management
Plus de 6 mots,111,28.3,Continuous Integration and Delivery for Mobile...


Un pitch Launch HN est très court : six mots en médiane, une quarantaine de caractères, d'autant que
les titres Hacker News sont limités à 80 caractères. Un tiers
nomme son client, mais les chiffres, les tournures « X for Y » et les mots à la mode y sont rares
(moins de 4 % chacun) : sur ces traits, les comparaisons reposeront sur une dizaine de startups et
seront forcément imprécises. La tournure « X for Y » est en outre sous-estimée, car seules les
marques de la liste sont reconnues (« Heroku for Deep Learning » échappe au repérage), et la forme « the X of Y » prend quelques faux
positifs (« with the help of live actors »). De même, le mot qui suit « for » n'est pas toujours un
client (« for Deep Learning »).

### 3.2 Le jargon suit l'époque

Avant de relier un mot à l'échec, il faut savoir s'il dépend de l'époque. On suit la part des pitchs
qui contiennent quelques mots à la mode, sur toutes les promotions depuis 2017 : ici, l'échec n'entre
pas en jeu, donc le manque de recul des promotions récentes ne pose pas de problème.

In [18]:
annee_table = table["promotion"].map(nettoyage.debut_promotion).dt.year
depuis_2017 = annee_table >= 2017
mots_pitchs = table.loc[depuis_2017, "pitch_origine"].map(texte.nettoyer_pitch).str.split()
MOTS_A_LA_MODE = ["ai", "platform", "marketplace", "crypto"]
parts = (
    pd.DataFrame({mot: mots_pitchs.map(lambda mots, mot=mot: mot in mots) for mot in MOTS_A_LA_MODE})
    .groupby(annee_table[depuis_2017])
    .mean()
)
jargon_par_annee = table.loc[depuis_2017, "pitch_origine"].map(texte.score_jargon).groupby(
    annee_table[depuis_2017]
).mean()
print("Score de jargon moyen par année (part des mots du pitch qui sont à la mode) :")
print(jargon_par_annee.round(3).to_string())
graphiques.courbe_mots_par_annee(parts, "« AI » est passé de 2 % à un tiers des pitchs en quatre ans")

Score de jargon moyen par année (part des mots du pitch qui sont à la mode) :
promotion
2017    0.003
2018    0.008
2019    0.006
2020    0.002
2021    0.005
2022    0.022
2023    0.041
2024    0.055
2025    0.068
2026    0.062


« AI » apparaît dans 1 à 2 % des pitchs des promotions 2017 à 2021, puis 5 % en 2022, 22 % en 2023
et environ un tiers depuis 2024 ; le score de jargon suit la même courbe (0,003 en 2017, 0,068 en
2025). « Platform » culmine en 2022, « crypto » et « marketplace » restent rares. Le vocabulaire d'un
pitch dit donc d'abord de quelle année il date. Conséquence pour la suite : dans l'échantillon
2017-2021, les mots à la mode sont presque absents (8 pitchs), et on ne peut pas tester si le pitch
« AI » des promotions récentes porte malheur.

### 3.3 Les traits du pitch et l'échec, à promotion égale

Pour chaque trait, on compare les échecs observés parmi les startups qui le présentent aux échecs
attendus si chacune échouait comme sa promotion (`analyse.ratios_par_indicateur`, même méthode
qu'en 2.3). Un ratio de 1,5 se lit « 50 % d'échecs de plus que leurs promotions ».

In [19]:
ratios = analyse.ratios_par_indicateur(etude, list(INDICATEURS)).rename(index=INDICATEURS)
display(ratios.round(2))
graphiques.points_ratios(ratios, "Aucun trait du pitch ne change nettement le risque d'échec")

,effectif,observes,attendus,ratio,ic_bas,ic_haut
indicateur,,,,,,
Contient un chiffre,14,4,3.19,1.25,0.34,3.21
Nomme un client (« for … »),130,32,29.95,1.07,0.73,1.51
Tournure « X for Y »,8,4,1.68,2.38,0.64,6.10
Contient un mot à la mode,8,4,1.87,2.14,0.58,5.48
Plus de 6 mots,111,27,25.98,1.04,0.68,1.51


Tous les intervalles contiennent 1 : aucun trait du pitch n'est associé à un risque d'échec
différent de celui de sa promotion. Les pitchs qui nomment leur client (130 startups) ou qui sont
plus longs que la médiane (111) échouent comme les autres (ratios de 1,07 et 1,04). Les tournures
« X for Y » et les mots à la mode penchent vers plus d'échecs (4 échecs sur 8 pitchs chacun, ratios
autour de 2), mais sur si peu de startups l'intervalle va de 0,6 à 6 : c'est une piste, pas un
résultat.

### 3.4 Les mots du succès et de l'échec

On cherche enfin, mot par mot, ceux qui sont plus fréquents chez les startups qui n'ont pas échoué
(`texte.mots_associes`, log-odds ratio). Le nom de la startup est d'abord masqué
(`texte.masquer_nom`) pour ne pas compter comme un mot. Avec des dizaines de mots testés, quelques-uns
paraissent liés à l'échec par simple hasard : les p-valeurs sont corrigées par la méthode de
Benjamini-Hochberg, et seuls les mots présents dans au moins 5 pitchs sont gardés.

In [20]:
etude["pitch_masque"] = [
    texte.masquer_nom(pitch, nom) for pitch, nom in zip(etude["pitch_origine"], etude["nom"])
]
mots = texte.mots_associes(etude["pitch_masque"], 1 - etude["echec"], freq_min=5)
print(f"{len(mots)} mots présents dans au moins 5 pitchs ; plus petite p-valeur : "
      f"{mots['p_valeur'].min():.2f}, après correction : {mots['p_ajustee'].min():.2f}")
colonnes = ["frequence", "nb_succes", "nb_echec", "log_odds_ratio", "p_valeur", "p_ajustee"]
display(pd.concat([mots.nlargest(5, "z"), mots.nsmallest(5, "z")])[colonnes].round(2))
graphiques.mots_frequence_association(
    mots, "Aucun mot du pitch n'est associé au succès ou à l'échec après correction"
)

58 mots présents dans au moins 5 pitchs ; plus petite p-valeur : 0.06, après correction : 0.78


,frequence,nb_succes,nb_echec,log_odds_ratio,p_valeur,p_ajustee
mot,,,,,,
software,15,14,1,1.10,0.20,0.78
team,19,17,2,0.77,0.26,0.78
automated,7,7,0,1.52,0.30,0.78
find,7,7,0,1.52,0.30,0.78
tool,7,7,0,1.52,0.30,0.78
marketplace,5,2,3,-1.57,0.06,0.78
real,6,3,3,-1.23,0.11,0.78
online,16,10,6,-0.76,0.14,0.78
people,10,6,4,-0.86,0.17,0.78


Aucun mot ne se distingue : la plus petite p-valeur dépasse déjà 0,05 avant correction, et toutes
les p-valeurs corrigées sont au-dessus de 0,7. Les mots les plus « techniques » (software, team,
tool) penchent côté succès et des mots grand public (online, people, marketplace) côté échec, ce
qui rappelle l'écart entre B2B et Consumer de la partie 2 plus qu'un effet du pitch lui-même. C'est
un résultat nul, et il est cohérent avec la partie 3.3 : sur 392 startups, le vocabulaire du pitch
d'origine ne distingue pas celles qui échoueront.

### 3.5 Les gagnants réécrivent-ils leur pitch ?

Le pitch actuel de la base YC n'est pas une variable explicative (partie 1.2), mais l'écart entre
les deux pitchs dit quelque chose : une startup qui a pivoté ou grandi réécrit sa phrase.
`texte.similarite_pitchs` mesure la part de mots communs aux deux pitchs (indice de Jaccard, de 0 :
aucun mot commun, à 1 : mêmes mots), après avoir masqué le nom de la startup.

In [21]:
pitch_actuel_masque = [
    texte.masquer_nom(pitch, nom) for pitch, nom in zip(etude["pitch_actuel"], etude["nom"])
]
etude["similarite"] = texte.similarite_pitchs(
    etude["pitch_masque"], pd.Series(pitch_actuel_masque, index=etude.index)
)
etude["reecrit"] = etude["similarite"] == 0

similarite = etude.groupby("statut")["similarite"].agg(
    startups="size", moyenne="mean", mediane="median"
)
similarite["aucun mot commun (%)"] = etude.groupby("statut")["reecrit"].mean() * 100
# À promotion égale : écart à la similarité moyenne de la promotion.
ecart = etude["similarite"] - etude.groupby("promotion")["similarite"].transform("mean")
similarite["écart à la promotion"] = ecart.groupby(etude["statut"]).mean()
display(similarite.round(2))
analyse.ratios_par_indicateur(etude, ["reecrit"]).rename(
    index={"reecrit": "Pitch entièrement réécrit"}
).round(2)

,startups,moyenne,mediane,aucun mot commun (%),écart à la promotion
statut,,,,,
Acquired,87,0.19,0.11,39.08,0.01
Active,215,0.16,0.10,41.40,-0.02
Inactive,90,0.21,0.10,44.44,0.03


,effectif,observes,attendus,ratio,ic_bas,ic_haut
indicateur,,,,,,
Pitch entièrement réécrit,163,40,39.32,1.02,0.73,1.39


In [22]:
exemples_reecriture = ["Hightouch", "Substack", "Voodoo Manufacturing"]
etude.loc[
    etude["nom"].isin(exemples_reecriture),
    ["nom", "statut", "pitch_origine", "pitch_actuel", "similarite"],
].round(2)

,nom,statut,pitch_origine,pitch_actuel,similarite
3,Voodoo Manufacturing,Inactive,AWS for Manufacturing,AWS for manufacturing.,1.0
57,Substack,Active,Paid email newsletters made simple,A place for independent writing.,0.0
134,Hightouch,Active,We Book Travel for You on Slack,The leading Composable CDP and AI Decisioning ...,0.0


La réécriture est la règle : en médiane, un dixième seulement des mots est commun aux deux pitchs,
et plus de 40 % des startups n'en gardent aucun. Elle ne sépare pourtant pas les destins : les startups
rachetées, actives ou inactives ont des similarités proches (0,19, 0,16 et 0,21 en moyenne), et les
pitchs entièrement réécrits échouent autant que leur promotion (ratio de 1,02, intervalle de 0,73 à
1,39). Les inactives gardent même un peu plus leur pitch d'origine, ce qui s'explique sans doute par
une fiche YC qui n'est plus mise à jour après l'arrêt (Voodoo Manufacturing est restée « AWS for
manufacturing ») plutôt que par un pitch mieux trouvé.

### 3.6 Et la foule ? Réaction sur Hacker News

Le post Launch HN reçoit des votes (points) et des commentaires. Les lecteurs de Hacker News
repèrent-ils les startups qui vont durer ? `analyse.reaction_hn_et_destin` décrit points et
commentaires par statut, et on compare le taux d'échec par quart de points.

In [23]:
display(analyse.reaction_hn_et_destin(etude))
etude["quart_points"] = pd.qcut(
    etude["hn_points"], 4, labels=["1er quart", "2e quart", "3e quart", "4e quart"]
)
display(analyse.taux_echec_standardise(etude, "quart_points").sort_index().round(2))
graphiques.boites_reaction(
    etude, "Les startups qui échoueront ont reçu autant de votes que les autres"
)

,startups,points_q1,points_mediane,points_q3,commentaires_q1,commentaires_mediane,commentaires_q3
statut,,,,,,,
Acquired,87,69.00,93.0,141.50,32.5,45.0,75.00
Active,215,70.00,104.0,164.00,36.5,59.0,88.00
Inactive,90,68.75,113.0,165.75,41.0,62.0,117.25


,effectif,observes,attendus,ratio,ic_bas,ic_haut
quart_points,,,,,,
1er quart,100,23,22.44,1.02,0.65,1.54
2e quart,97,16,23.17,0.69,0.39,1.12
3e quart,97,26,22.59,1.15,0.75,1.69
4e quart,98,25,21.80,1.15,0.74,1.69


La réaction de Hacker News ne prédit pas le destin : la médiane des points est de 93 pour les
startups rachetées, 104 pour les actives et 113 pour les inactives, avec des boîtes qui se recouvrent
presque entièrement ; les commentaires aussi (médianes de 45, 59 et 62). Par quart de points, aucun ratio
d'échec à promotion égale ne s'écarte de 1 de façon significative. Les startups les plus commentées
ne sont pas plus solides : un post très discuté peut l'être pour de mauvaises raisons.

## 4. Le pitch dit-il plus que le secteur et l'époque ?

La seconde hypothèse de l'introduction dit que le pitch ne fait que refléter le secteur et l'époque.
La partie 2 a montré que le secteur sépare déjà fortement les startups (Consumer échoue 2,2 fois
plus qu'attendu), et un trait du pitch peut simplement signaler un secteur : un pitch « X for Y »
décrit souvent une application grand public. On reprend donc les ratios de la partie 3.3, mais
cette fois à promotion **et** secteur égaux.

### 4.1 Méthode : comparer à secteur et promotion égaux

Chaque startup reçoit désormais le taux d'échec de sa case promotion × secteur (`strates` de
`analyse.taux_echec_standardise`) ; la somme de ces taux donne les échecs attendus. Si le ratio
d'un trait revient vers 1 quand on ajoute le secteur, son effet venait du secteur ; s'il tient, le
pitch dit quelque chose de plus. On préfère cette comparaison à un modèle de régression parce
qu'elle ne suppose aucune forme pour l'effet du secteur ou de la promotion, et que son résultat se
lit directement : « x % d'échecs de plus que prévu ».

Sa limite tient au nombre de cases : une startup seule dans sa case a autant d'échecs attendus
qu'observés et n'apporte rien à la comparaison. Les échecs attendus sont en outre calculés sur toutes
les startups de la case, y compris celles qui présentent le trait : pour un trait fréquent, le ratio
est un peu tassé vers 1.

In [24]:
cases = etude.groupby(["promotion", "secteur"])["slug"].transform("size")
print(f"{etude.groupby(['promotion', 'secteur']).ngroups} cases promotion × secteur ; "
      f"{(cases >= 5).mean():.0%} des startups sont dans une case d'au moins 5 startups, "
      f"{int((cases == 1).sum())} sont seules dans leur case")

61 cases promotion × secteur ; 81% des startups sont dans une case d'au moins 5 startups, 16 sont seules dans leur case


### 4.2 Résultat

In [25]:
COMPARAISONS = {
    "À promotion égale": ("promotion",),
    "À promotion et secteur égaux": ("promotion", "secteur"),
}
comparaison = pd.concat(
    [
        analyse.ratios_par_indicateur(etude, list(INDICATEURS), strates)
        .rename(index=INDICATEURS)
        .assign(comparaison=nom)
        for nom, strates in COMPARAISONS.items()
    ]
)
display(
    comparaison.pivot(columns="comparaison", values=["ratio", "ic_bas", "ic_haut"])
    .reindex(INDICATEURS.values())
    .reindex(columns=list(COMPARAISONS), level=1)
    .round(2)
)
graphiques.points_ratios(
    comparaison,
    "À secteur égal, aucun trait du pitch ne change le risque d'échec",
    axe_x="Échecs observés / attendus (échelle log)",
)

ratio                               \
comparaison                 À promotion égale À promotion et secteur égaux   
indicateur                                                                   
Contient un chiffre                      1.25                         1.08   
Nomme un client (« for … »)              1.07                         1.10   
Tournure « X for Y »                     2.38                         1.36   
Contient un mot à la mode                2.14                         1.90   
Plus de 6 mots                           1.04                         1.03   

                                       ic_bas                               \
comparaison                 À promotion égale À promotion et secteur égaux   
indicateur                                                                   
Contient un chiffre                      0.34                         0.29   
Nomme un client (« for … »)              0.73                         0.75   
Tournure « X for Y »                     0.64                         0.36   
Contient un mot à la mode                0.58                         0.51   
Plus de 6 mots                           0.68                         0.68   

                                      ic_haut                               
comparaison                 À promotion égale À promotion et secteur égaux  
indicateur                                                                  
Contient un chiffre                      3.21                         2.76  
Nomme un client (« for … »)              1.51                         1.55  
Tournure « X for Y »                     6.10                         3.47  
Contient un mot à la mode                5.48                         4.86  
Plus de 6 mots                           1.51                         1.50

Les 81 % de startups rangées dans une case d'au moins cinq portent l'essentiel de la comparaison.
Ajouter le secteur ne fait apparaître aucun effet : tous les intervalles contiennent encore 1. Le
ratio de la tournure « X for Y » passe de 2,4 à 1,4 : une partie de son excès d'échecs venait du
secteur. Celui des mots à la mode reste vers 1,9, mais sur 8 startups (intervalle de 0,5 à 4,9).
Les deux traits fréquents ne bougent pas : nommer son client (1,10) et écrire un pitch long (1,03)
vont avec autant d'échecs que prévu.

La réponse à la question est donc négative sur ces données : au-delà du secteur et de l'époque,
aucun trait mesurable du pitch d'origine n'est associé à l'échec. Ce résultat nul a une précision
connue, à lire avec prudence : pour les startups qui nomment leur client, l'intervalle va de 0,75 à
1,55, et il serait un peu plus large face aux seules startups qui ne nomment pas de client (partie
4.1). On peut écarter un risque d'échec doublé, pas un effet modéré. Les traits rares, eux, ne
peuvent être ni confirmés ni écartés.

### 4.3 Robustesse

Le résultat tient-il si l'on change les choix de la partie 1 ? On refait la comparaison à promotion
et secteur égaux sur d'autres fenêtres de promotions (2017-2019, 2017-2020, 2017-2022, 2017-2023),
puis sur 2017-2021 en comptant les rachats de sauvetage (rachetées à cinq personnes ou moins) comme
des échecs. On regarde seulement si le sens des écarts tient d'une version à l'autre : les
fenêtres plus longues ajoutent des promotions qui ont eu moins de temps pour échouer.

In [26]:
def ratios_du_pitch(echantillon: pd.DataFrame) -> pd.Series:
    """Ratios à promotion et secteur égaux des traits du pitch, recalculés sur `echantillon`."""
    # Même seuil de longueur que dans la partie 3.1, pour comparer les versions entre elles.
    traits = texte.traits_pitch(echantillon["pitch_origine"], longueur_mediane)
    echantillon = echantillon.assign(**traits)
    ratios = analyse.ratios_par_indicateur(
        echantillon, list(INDICATEURS), strates=("promotion", "secteur")
    )
    return ratios["ratio"].rename(index=INDICATEURS)


echantillons = {
    f"2017-{fin}": nettoyage.echantillon_etude(table, debut=2017, fin=fin)[0]
    for fin in [2019, 2020, 2021, 2022, 2023]
}
sauvetage = succes.rachats_de_sauvetage(etude)
echantillons["2017-2021, sauvetages = échecs"] = etude.assign(
    statut=etude["statut"].where(~sauvetage, "Inactive")
)

robustesse = pd.DataFrame(
    {nom: ratios_du_pitch(echantillon) for nom, echantillon in echantillons.items()}
).T.round(2)
robustesse.insert(0, "startups", [len(e) for e in echantillons.values()])
robustesse.insert(1, "échecs", [succes.echec(e).sum() for e in echantillons.values()])
robustesse.rename_axis("version")

indicateur,startups,échecs,Contient un chiffre,Nomme un client (« for … »),Tournure « X for Y »,Contient un mot à la mode,Plus de 6 mots
version,,,,,,,
2017-2019,167,43,0.77,1.15,1.00,0.97,0.87
2017-2020,244,59,0.77,1.26,0.70,1.57,0.99
2017-2021,392,90,1.08,1.10,1.36,1.90,1.03
2017-2022,467,101,1.09,1.16,1.51,1.60,0.99
2017-2023,525,109,1.05,1.12,1.43,1.30,1.01
"2017-2021, sauvetages = échecs",392,122,0.90,0.98,1.24,1.82,0.99


Les pitchs longs restent entre 0,87 et 1,03 dans toutes les versions. Ceux qui nomment leur client
sont un peu au-dessus de 1 dans cinq versions sur six (jusqu'à 1,26), mais toujours dans la marge
d'incertitude de la partie 4.2 : c'est la seule tendance un peu régulière, trop faible pour
conclure. Les traits rares bougent beaucoup d'une version à l'autre (de 0,70 à 1,51 pour
« X for Y », de 0,97 à 1,90 pour les mots à la mode), ce qui confirme qu'on ne peut rien en
conclure. Compter les 32 rachats de sauvetage comme des échecs ne change pas la lecture. Le
résultat nul de la partie 4.2 ne dépend donc ni de la fenêtre 2017-2021 ni de la définition stricte
de l'échec.

## Conclusion

Sur 392 startups YC des promotions 2017 à 2021, aucun trait mesurable du pitch d'origine n'est
associé à l'échec une fois la promotion et le secteur pris en compte : les startups qui nomment leur
client échouent 1,10 fois autant que prévu (intervalle de 0,75 à 1,55), celles au pitch long 1,03
fois, et aucun mot ne se distingue. Ni la réaction des lecteurs de Hacker News ni la réécriture
ultérieure du pitch ne séparent davantage les destins. La seconde hypothèse l'emporte : ce qui
distingue les startups, c'est d'abord le secteur (le grand public échoue 2,2 fois plus qu'attendu)
et l'époque, que le pitch reflète, plutôt que la manière dont il est écrit. On ne peut donc pas,
sur ces données, prédire le succès d'une startup à partir de sa première phrase.

Ce résultat a plusieurs limites.

- **Le recul.** Même les promotions 2017 à 2021 perdent encore des startups, et le statut *Active*
  ne dit pas si l'entreprise grandit ou survit à peine. Le statut vient en outre de la base YC, qui
  n'est pas toujours tenue à jour après un arrêt (partie 3.5).
- **La représentativité.** Une startup YC sur cinq seulement publie un Launch HN, et le B2B y est
  surreprésenté (55 % contre 42 %). Toutes ont aussi été choisies par YC, qui écarte sans doute déjà
  les pitchs les plus faibles : les écarts entre pitchs retenus sont forcément plus petits.
- **Des contrôles mesurés aujourd'hui.** Le secteur et la région sont ceux de la base YC actuelle,
  après d'éventuels pivots, et non ceux du lancement.
- **La puissance.** Le résultat nul est précis pour les traits fréquents (client nommé, longueur),
  mais les chiffres, les tournures « X for Y » et les mots à la mode concernent chacun une dizaine
  de startups : ils ne peuvent être ni confirmés ni écartés. La mesure elle-même est simple et
  bruitée : elle repose sur des listes de marques et de mots choisies à la main, prend quelques faux
  positifs (partie 3.1), et une mesure bruitée rapproche tous les ratios de 1, ce qui peut masquer un
  effet modéré.
- **Corrélation et causalité.** Même un lien aurait été une association : réécrire son pitch ne
  changerait pas le destin d'une startup.

Prédire le succès d'une startup demanderait des promotions plus anciennes, avec assez d'échecs pour
tester une prédiction : ce n'est pas l'objet de ce rapport, qui décrit.

Deux pistes permettraient d'aller plus loin. La première allongerait le recul : les pitchs des
promotions antérieures à 2017, trop rares sur Launch HN, pourraient être retrouvés dans les
anciennes versions des pages YC archivées par la Wayback Machine, ou dans les autres posts Hacker
News des fondateurs. La seconde mesurerait le succès autrement que par un statut : les déclarations
de levée de fonds déposées auprès du régulateur boursier américain (formulaire D de la SEC) donnent,
pour beaucoup de startups, les montants levés au fil des années.

## Annexes

### A. Dictionnaire des variables de la table commune

Une ligne par startup appariée (`nettoyage.joindre`). Les descriptions viennent de
`config.DICTIONNAIRE_VARIABLES`, qui fixe aussi l'ordre des colonnes.

In [27]:
dictionnaire = pd.DataFrame(
    {"type": table.dtypes.astype(str), "description": pd.Series(DICTIONNAIRE_VARIABLES)}
).rename_axis("variable")
with pd.option_context("display.max_colwidth", None):
    display(dictionnaire)

,type,description
variable,,
slug,str,Identifiant unique de la startup dans la base YC
nom,str,Nom actuel de la startup dans la base YC
promotion,str,"Promotion YC (« Winter 2020 », « Summer 2021 »…)"
pitch_origine,str,"Pitch du post Launch HN, publié au lancement"
pitch_actuel,str,"Pitch actuel de la base YC (champ one_liner), réécrit au fil du temps"
statut,str,"Statut actuel dans la base YC : Active, Inactive (échec), Acquired ou Public"
niveau_succes,object,"Échec, survie, croissance ou sortie (succes.niveau_succes), calculé après la jointure"
secteur,str,"Secteur principal dans la base YC (B2B, Consumer, Fintech, Healthcare…)"
region,str,"Grande région de la base YC (America / Canada, Europe…), vide si aucune"


### B. Relancer le projet

Depuis un service VSCode ou Jupyter du [SSP Cloud](https://datalab.sspcloud.fr) (Python 3.10 ou
plus) :

```bash
git clone https://github.com/romainmugniot/projet-pitch-startups.git
cd projet-pitch-startups
pip install -e .
```

puis ouvrir `rapport.ipynb` et exécuter toutes les cellules (*Restart & Run All*). Le notebook lit
la copie du 30/10/2026 stockée sur le S3 du SSP Cloud, lisible par tout utilisateur connecté, et
télécharge au premier lancement les ressources NLTK dont il a besoin. Hors du SSP Cloud, ou avec
`COLLECTER = True`, il retélécharge les données depuis les deux API publiques, sans clé : les
chiffres peuvent alors différer un peu du texte. Les tests des fonctions de `src/` se lancent avec
`pip install -e ".[dev]"` puis `pytest`.